In [0]:
from src.data_quality import (
    row_count,
    null_count,
    duplicate_count,
    invalid_quantity_count,
    invalid_price_count,
    invalid_discount_count,
    invalid_status_count
)
import pyspark.sql.functions as F

In [0]:
SILVER_TABLE = "workspace.silver.orders"

silver_df = spark.table(SILVER_TABLE)

print("Silver records:", silver_df.count())

In [0]:
total_records = row_count(silver_df)

print("Total records:", total_records)

In [0]:
null_customers = null_count(
    silver_df,
    "customer_id"
)

print("Null customers:", null_customers)

In [0]:
duplicate_orders = duplicate_count(
    silver_df,
    "order_id"
)

print(
    "Duplicate order ID groups:",
    duplicate_orders
)

In [0]:
invalid_quantities = invalid_quantity_count(
    silver_df
)

print(
    "Invalid quantities:",
    invalid_quantities
)

In [0]:
invalid_prices = invalid_price_count(
    silver_df
)

print(
    "Invalid prices:",
    invalid_prices
)

In [0]:
invalid_discounts = invalid_discount_count(
    silver_df
)

print(
    "Invalid discounts:",
    invalid_discounts
)

In [0]:
invalid_statuses = invalid_status_count(
    silver_df
)

print(
    "Invalid statuses:",
    invalid_statuses
)

In [0]:
dq_results = [
    (
        "CUSTOMER_ID_NOT_NULL",
        "ERROR",
        null_customers
    ),
    (
        "UNIQUE_ORDER_ID",
        "ERROR",
        duplicate_orders
    ),
    (
        "QUANTITY_GREATER_THAN_ZERO",
        "ERROR",
        invalid_quantities
    ),
    (
        "UNIT_PRICE_NON_NEGATIVE",
        "ERROR",
        invalid_prices
    ),
    (
        "DISCOUNT_BETWEEN_0_AND_100",
        "ERROR",
        invalid_discounts
    ),
    (
        "VALID_ORDER_STATUS",
        "ERROR",
        invalid_statuses
    )
]

In [0]:
dq_schema = """
rule_name STRING,
severity STRING,
failed_records LONG
"""

dq_df = spark.createDataFrame(
    dq_results,
    schema=dq_schema
)

In [0]:
import pyspark.sql.functions as F

dq_df = (
    dq_df
    .withColumn(
        "status",
        F.when(
            F.col("failed_records") == 0,
            "PASS"
        ).otherwise("FAIL")
    )
    .withColumn(
        "check_timestamp",
        F.current_timestamp()
    )
)

In [0]:
display(dq_df)

In [0]:
dq_df = (
    dq_df
    .withColumn(
        "batch_id",
        F.lit("CURRENT")
    )
)

In [0]:
DQ_TABLE = "workspace.audit.data_quality_results"

In [0]:
(
    dq_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(DQ_TABLE)
)

In [0]:
display(
    spark.table(DQ_TABLE)
)

In [0]:
failed_error_checks = (
    dq_df
    .filter(
        (F.col("severity") == "ERROR")
        & (F.col("status") == "FAIL")
    )
    .count()
)

if failed_error_checks > 0:
    raise Exception(
        f"Data Quality FAILED: "
        f"{failed_error_checks} critical rule(s) failed."
    )

print("Data Quality PASSED")

In [0]:
test_bad_df = (
    silver_df
    .limit(1)
    .withColumn(
        "quantity",
        F.lit(-10)
    )
)

In [0]:
test_invalid_quantity = invalid_quantity_count(
    test_bad_df
)

print(
    "Invalid quantity count:",
    test_invalid_quantity
)

In [0]:
if test_invalid_quantity > 0:
    print("TEST PASSED: DQ correctly detected invalid quantity.")
else:
    print("TEST FAILED")